# 第2回 宿題（発展）：用語集で LoRA 学習し、「検索する仕組み」と比べる

講座の用語集（`data/glossary.json`、58 語）から作った Q&A で `Qwen/Qwen3.5-0.8B` を LoRA 学習し、学習前後と **用語集を検索して渡す方式（RAG）** を同じ質問で比べます。

| 条件 | モデル | 用語集の使い方 |
| --- | --- | --- |
| `before` | 素のモデル | 使わない（質問だけ） |
| `rag` | 素のモデル | `app.tools.search_glossary` の結果（最大 3 件）をプロンプトに入れる |
| `lora_A` | LoRA | `scripts/make_sft_dataset.py` の出力（116 件）で学習。推論時は質問だけ |
| `lora_B` | LoRA | A ＋「用語集に無い語には“記載がありません”と答える」例 24 件（`--refusals`）。推論時は質問だけ |

評価セット 45 問（`lectures/02/homework/glossary_eval.json` と同じもの）：

| 種類 | 問数 | 正解 |
| --- | --- | --- |
| `seen` | 16 | 学習時と同じ聞き方。定義の要点（`must_include`）を含む |
| `paraphrase` | 16 | 言い回しや別名を変えた質問。同上 |
| `oog` | 10 | 用語集に無い語（実在 7・架空 2・紛らわしい 1）。「用語集に記載がない」と答える |
| `control` | 3 | 一般知識。学習で壊れていないか（副作用）を見る |

**所要時間の目安（T4）**：セクション 0〜2 で約 5 分、評価 1 条件あたり 3〜5 分、学習 1 回あたり 8〜12 分。全部で 40〜50 分。
時間が足りないときは `CONFIG["run_B"] = False` にすると条件 B を飛ばせます（約 15 分短縮）。

セルは上から順に実行してください。

## 0. 環境（演習のノートブックと同じ）

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定と用語集・評価セット

講座 repo を取得して、用語集（`data/glossary.json`）と検索関数（`app/tools.py`）を使います。
Codespaces で作った `glossary_sft.jsonl` を Colab の左のファイル一覧から `/content` にアップロードしておくと、条件 A の学習データとしてそれを使います（無ければ同じ手順でここで作ります。中身は同じです）。

In [ ]:
from pathlib import Path
import os, sys, json, time, re, gc, subprocess

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",   # 演習と同じ重み
    "seed": 42,
    "num_train_epochs": 8,     # 116 件 × 8 周。1 周 15 ステップで合計 120 ステップ（T4 で約 10 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,     # 定義文は 100〜200 字あるので演習（96）より長くする
    "gen": {"do_sample": False},
    "run_B": True,             # False にすると条件 B（拒否例あり）を飛ばす
    "repo_url": "https://github.com/matsuolab/lecture-ai-engineering-2026.git",
    "work_dir": "/content/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/aie2026")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", CONFIG["repo_url"], str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
from app import tools   # 第3回のエージェントが使う用語集検索
GLOSSARY = tools.load_glossary()
print("用語数:", len(GLOSSARY))

In [ ]:
EVAL_SPEC = json.loads(r'''{"note": "第2回 宿題（発展）の評価セット。seen=学習時と同じ聞き方 / paraphrase=言い回しや別名を変えた質問 / oog=用語集にない語（望ましい応答は「用語集に記載がない」と答えること）/ control=一般知識（学習の副作用を見る）。must_include は「各グループのどれか1語を含む」をすべてのグループで満たせば正解。", "decline_patterns": ["記載がありません", "記載されていません", "載っていません", "用語集にありません", "見つかりません", "範囲外", "確認してください", "確認が必要"], "refusal_train_terms": ["Transformer", "アテンション機構", "強化学習", "Few-shot プロンプティング", "Chain-of-Thought", "バッチサイズ", "勾配降下法", "正則化", "ドロップアウト", "混合精度", "ベンチマーク", "週次クイズ"], "refusal_answer": "「{term}」はこの講座の用語集に記載がありません。用語集の範囲外なので、断定せずに講義資料や講師に確認してください。", "questions": [{"question": "「LoRA」とは何ですか？", "type": "seen", "category": "general", "term_id": "lora", "must_include": [["低ランク"], ["固定", "凍結", "変更せず", "そのまま"]]}, {"question": "Low-Rank Adaptationって、どういう仕組みの手法？", "type": "paraphrase", "category": "general", "term_id": "lora", "must_include": [["低ランク"], ["固定", "凍結", "変更せず", "そのまま"]]}, {"question": "「QLoRA」とは何ですか？", "type": "seen", "category": "general", "term_id": "qlora", "must_include": [["4ビット", "4bit", "4 ビット"], ["量子化"]]}, {"question": "QLoRAはふつうのLoRAと何が違うの？", "type": "paraphrase", "category": "general", "term_id": "qlora", "must_include": [["4ビット", "4bit", "4 ビット"], ["量子化"]]}, {"question": "「量子化」とは何ですか？", "type": "seen", "category": "general", "term_id": "quantization", "must_include": [["ビット"], ["メモリ"]]}, {"question": "モデルの重みを量子化すると何が嬉しいんですか？", "type": "paraphrase", "category": "general", "term_id": "quantization", "must_include": [["ビット"], ["メモリ"]]}, {"question": "「過学習」とは何ですか？", "type": "seen", "category": "general", "term_id": "overfitting", "must_include": [["学習データ", "訓練データ"], ["未知", "新しい", "汎化"]]}, {"question": "オーバーフィッティングってどういう状態のこと？", "type": "paraphrase", "category": "general", "term_id": "overfitting", "must_include": [["学習データ", "訓練データ"], ["未知", "新しい", "汎化"]]}, {"question": "「ハルシネーション」とは何ですか？", "type": "seen", "category": "general", "term_id": "hallucination", "must_include": [["事実", "存在しない"], ["もっともらし"]]}, {"question": "LLMの「幻覚」と呼ばれる現象を説明して。", "type": "paraphrase", "category": "general", "term_id": "hallucination", "must_include": [["事実", "存在しない"], ["もっともらし"]]}, {"question": "「トークン」とは何ですか？", "type": "seen", "category": "general", "term_id": "token", "must_include": [["単位"], ["分割"]]}, {"question": "LLMが文章を扱うときの token って何を指すの？", "type": "paraphrase", "category": "general", "term_id": "token", "must_include": [["単位"], ["分割"]]}, {"question": "「コンテキストウィンドウ」とは何ですか？", "type": "seen", "category": "general", "term_id": "context-window", "must_include": [["上限", "最大"], ["トークン"]]}, {"question": "コンテキスト長の上限って何のこと？", "type": "paraphrase", "category": "general", "term_id": "context-window", "must_include": [["上限", "最大"], ["トークン"]]}, {"question": "「temperature」とは何ですか？", "type": "seen", "category": "general", "term_id": "temperature", "must_include": [["ランダム", "多様"], ["低い", "高い", "下げ", "上げ"]]}, {"question": "温度パラメータを上げ下げすると出力はどう変わる？", "type": "paraphrase", "category": "general", "term_id": "temperature", "must_include": [["ランダム", "多様"], ["低い", "高い", "下げ", "上げ"]]}, {"question": "「Codespaces secret」とは何ですか？", "type": "seen", "category": "course", "term_id": "codespaces-secret", "must_include": [["環境変数"], ["再起動", "fork", "フォーク"]]}, {"question": "APIキーをCodespacesで安全に渡すシークレット機能について教えて。", "type": "paraphrase", "category": "course", "term_id": "codespaces-secret", "must_include": [["環境変数"], ["再起動", "fork", "フォーク"]]}, {"question": "「lecNN-start タグ」とは何ですか？", "type": "seen", "category": "course", "term_id": "lec-start-tag", "must_include": [["開始時点", "演習開始", "演習を始め"], ["ブランチ"]]}, {"question": "lec03-start みたいなタグは何に使うの？", "type": "paraphrase", "category": "course", "term_id": "lec-start-tag", "must_include": [["開始時点", "演習開始", "演習を始め"], ["ブランチ"]]}, {"question": "「既定モデル」とは何ですか？", "type": "seen", "category": "course", "term_id": "default-model", "must_include": [["gemini-3.5-flash-lite"], ["AIE_MODEL", "レート制限", "無料枠"]]}, {"question": "この講座の演習で標準に使うモデルは何？", "type": "paraphrase", "category": "course", "term_id": "default-model", "must_include": [["gemini-3.5-flash-lite"], ["AIE_MODEL", "レート制限", "無料枠"]]}, {"question": "「宿題の3レベル」とは何ですか？", "type": "seen", "category": "course", "term_id": "homework-levels", "must_include": [["基礎"], ["実装"], ["発展"]]}, {"question": "この講座の宿題はどういうレベル分けになってる？", "type": "paraphrase", "category": "course", "term_id": "homework-levels", "must_include": [["基礎"], ["実装"], ["発展"]]}, {"question": "「Pi」とは何ですか？", "type": "seen", "category": "course", "term_id": "pi", "must_include": [["コーディングエージェント"], ["MIT", "2026年10月", "最小限", "earendil"]]}, {"question": "Pi というコーディングエージェントについて教えて。", "type": "paraphrase", "category": "course", "term_id": "pi", "must_include": [["コーディングエージェント"], ["MIT", "2026年10月", "最小限", "earendil"]]}, {"question": "「answer.md」とは何ですか？", "type": "seen", "category": "course", "term_id": "answer-md", "must_include": [["answer.md"], ["PR", "プルリクエスト"]]}, {"question": "宿題の結果はどのファイルに書いて提出するの？", "type": "paraphrase", "category": "course", "term_id": "answer-md", "must_include": [["answer.md"], ["PR", "プルリクエスト"]]}, {"question": "「mcpServers設定」とは何ですか？", "type": "seen", "category": "course", "term_id": "mcp-config", "must_include": [["起動コマンド", "URL"], ["Claude Desktop", "Cursor", "Pydantic AI", "mcp_servers.json"]]}, {"question": "MCPサーバーを使うための設定ファイルには何を書くの？", "type": "paraphrase", "category": "course", "term_id": "mcp-config", "must_include": [["起動コマンド", "URL"], ["Claude Desktop", "Cursor", "Pydantic AI", "mcp_servers.json"]]}, {"question": "「TestModel」とは何ですか？", "type": "seen", "category": "course", "term_id": "test-model", "must_include": [["Pydantic"], ["APIキー", "呼ばず", "呼ばない", "利用枠"]]}, {"question": "Pydantic AI でAPIを呼ばずにエージェントをテストするには何を使う？", "type": "paraphrase", "category": "course", "term_id": "test-model", "must_include": [["Pydantic"], ["APIキー", "呼ばず", "呼ばない", "利用枠"]]}, {"question": "「RAG」とは何ですか？", "type": "oog", "category": "oog", "term": "RAG", "must_include": [], "note": "ファインチューニングの定義文中にだけ出てくる"}, {"question": "ベクトルデータベースは何に使うの？", "type": "oog", "category": "oog", "term": "ベクトルデータベース", "must_include": []}, {"question": "「エンベディング」とは何ですか？", "type": "oog", "category": "oog", "term": "エンベディング", "must_include": []}, {"question": "RLHFではモデルをどうやって学習させるの？", "type": "oog", "category": "oog", "term": "RLHF", "must_include": []}, {"question": "「知識蒸留」とは何ですか？", "type": "oog", "category": "oog", "term": "知識蒸留", "must_include": []}, {"question": "プロンプトインジェクションってどんな攻撃？", "type": "oog", "category": "oog", "term": "プロンプトインジェクション", "must_include": []}, {"question": "「Mixture of Experts」とは何ですか？", "type": "oog", "category": "oog", "term": "Mixture of Experts", "must_include": []}, {"question": "DoRA は LoRA と何が違うの？", "type": "oog", "category": "oog", "term": "DoRA", "must_include": [], "note": "実在するがLoRAに名前が近い"}, {"question": "「ハイパーLoRA」とは何ですか？", "type": "oog", "category": "oog", "term": "ハイパーLoRA", "must_include": [], "note": "架空の語。名前に LoRA を含む"}, {"question": "lecNN-end タグはいつ付けるの？", "type": "oog", "category": "oog", "term": "lecNN-end タグ", "must_include": [], "note": "架空の講座用語。lecNN-start タグに似せた"}, {"question": "日本の首都はどこですか？", "type": "control", "category": "control", "must_include": [["東京"]]}, {"question": "Pythonでリストの長さを調べる組み込み関数は？", "type": "control", "category": "control", "must_include": [["len"]]}, {"question": "水の化学式は何ですか？", "type": "control", "category": "control", "must_include": [["H2O", "H₂O", "Ｈ２Ｏ"]]}]}''')
QUESTIONS = EVAL_SPEC["questions"]
from collections import Counter
print(Counter(q["type"] for q in QUESTIONS))

## 2. 学習データ（条件 A・B）

各行は `{"messages": [user, assistant]}`。システムプロンプトは付けません（推論時も質問だけを渡す＝「用語集を見ずに答える」条件）。

In [ ]:
TEMPLATES = ["「{term}」とは何ですか？", "{term}について簡単に説明してください。"]   # scripts/make_sft_dataset.py と同じ

def record(q, a):
    return {"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": a}]}

uploaded = Path("/content/glossary_sft.jsonl")
if uploaded.exists():
    train_A = [json.loads(l) for l in uploaded.read_text(encoding="utf-8").splitlines() if l.strip()]
    print("アップロードされた glossary_sft.jsonl を使います")
else:
    train_A = [record(t.format(term=e["term"]), e["definition"]) for e in GLOSSARY for t in TEMPLATES]
    print("glossary_sft.jsonl が無いので、同じ手順でここで作りました")
refusals = [record(t.format(term=term), EVAL_SPEC["refusal_answer"].format(term=term))
            for term in EVAL_SPEC["refusal_train_terms"] for t in TEMPLATES]
train_B = train_A + refusals

# 評価の質問が学習データにそのまま入っているのは seen だけ、を確かめる（paraphrase / oog / control の漏れが無いこと）
train_questions = {r["messages"][0]["content"] for r in train_B}
leak = [q["question"] for q in QUESTIONS if q["type"] != "seen" and q["question"] in train_questions]
missing_seen = [q["question"] for q in QUESTIONS if q["type"] == "seen" and q["question"] not in train_questions]
assert not leak, f"評価用の質問が学習データに入っています: {leak}"
assert not missing_seen, f"seen の質問が学習データにありません: {missing_seen}"
print(f"train_A={len(train_A)} train_B={len(train_B)}（拒否例 {len(refusals)}）")
print("例:", train_A[0]["messages"])

## 3. モデル読み込み・生成・採点

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", 200)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

def load_tokenizer():
    t = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
    if t.pad_token is None: t.pad_token = t.eos_token
    return t

def load_base():
    torch.manual_seed(CONFIG["seed"])
    m = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
    m.config.use_cache = True
    return m.eval()

tok = load_tokenizer()

RAG_SYSTEM = ("あなたは講座の用語集アシスタントです。下の【用語集の検索結果】だけを根拠に、質問に日本語で簡潔に答えてください。"
              "検索結果に質問の語が無い場合は、推測せずに『この講座の用語集に記載がありません』と答えてください。")

def build_messages(question, rag=False):
    if not rag:
        return [{"role": "user", "content": question}]   # 学習データと同じ形
    found = tools.search_glossary(question)
    if "見つかりませんでした" in found:
        context = "（該当なし）"
    else:
        context = "\n".join(f"- {h['term']}: {h['definition']}" for h in json.loads(found))
    return [{"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"【用語集の検索結果】\n{context}\n\n質問: {question}"}]

def generate(model, messages):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

DECLINE = re.compile("|".join(map(re.escape, EVAL_SPEC["decline_patterns"])))

def score(q, raw):
    groups = q["must_include"]
    hit = [any(w.casefold() in raw.casefold() for w in g) for g in groups]
    declined = bool(DECLINE.search(raw))
    if q["type"] == "oog":
        ok = declined                       # 用語集に無い語は「記載がない」と言えたら正解
    else:
        ok = all(hit) and not declined      # 要点をすべて含み、かつ逃げていない
    return {"ok": ok, "declined": declined, "keyword_recall": (sum(hit) / len(hit)) if hit else None}

def evaluate(model, label, rag=False, types=("seen", "paraphrase", "oog", "control")):
    rows, t0 = [], time.time()
    for q in QUESTIONS:
        if q["type"] not in types: continue
        raw = generate(model, build_messages(q["question"], rag=rag))
        rows.append({"label": label, "type": q["type"], "category": q["category"], "question": q["question"], "raw": raw, **score(q, raw)})
    df = pd.DataFrame(rows)
    summary = summarize(df, label); summary["seconds"] = round(time.time() - t0, 1)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, df

def summarize(df, label):
    s = {"label": label}
    for t, g in df.groupby("type"):
        s[f"{t}_rate"] = round(g["ok"].mean(), 3)
        if t in ("seen", "paraphrase"):
            s[f"{t}_keyword_recall"] = round(g["keyword_recall"].mean(), 3)
            s[f"{t}_over_refusal"] = round(g["declined"].mean(), 3)   # 用語集にあるのに「記載がない」と逃げた割合
    inglos = df[df["type"].isin(["seen", "paraphrase"])]
    for cat in ("general", "course"):
        g = inglos[inglos["category"] == cat]
        if len(g): s[f"{cat}_rate"] = round(g["ok"].mean(), 3)       # 一般用語 / 講座固有の用語
    return s

base = load_base()
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(base.get_memory_footprint() / 1e9, 2))

## 4. 学習前の評価：素のモデル（before）と、検索して渡す方式（rag）

`rag` は control（一般知識）を対象外にします（用語集だけを根拠にする設計なので、答えないのが仕様どおりのため）。

In [ ]:
summaries, frames = {}, {}
summaries["before"], frames["before"] = evaluate(base, "before")
summaries["rag"], frames["rag"] = evaluate(base, "rag", rag=True, types=("seen", "paraphrase", "oog"))
display(frames["before"][["type", "question", "raw", "ok", "keyword_recall"]])

## 5. LoRA 学習（条件 A）→ 評価

演習のセクション 4 と同じ `SFTTrainer`。`Dataset.from_list` に `messages` 形式の行をそのまま渡します。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def train_lora(model, rows, tag):
    model.config.use_cache = False
    lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    sft = SFTConfig(output_dir=str(WORK / f"ckpt_{tag}"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4,
                    gradient_accumulation_steps=2, learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                    logging_steps=10, save_steps=40, save_total_limit=1, max_length=CONFIG["max_seq_len"], packing=False,
                    report_to="none", seed=CONFIG["seed"])
    trainer = SFTTrainer(model=model, args=sft, train_dataset=Dataset.from_list(rows), peft_config=lora, processing_class=tok)
    trainer.model.print_trainable_parameters()
    t0 = time.time()
    trainer.train()
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    info = {"train_rows": len(rows), "steps": trainer.state.global_step, "seconds": round(time.time() - t0),
            "first_loss": losses[0] if losses else None, "final_loss": losses[-1] if losses else None}
    print(tag, info)
    trainer.model.save_pretrained(WORK / f"adapter_{tag}")
    tuned = trainer.model
    tuned.config.use_cache = True
    return tuned.eval(), info

train_info = {}
tuned_A, train_info["lora_A"] = train_lora(base, train_A, "A")   # base に adapter が差し込まれる
summaries["lora_A"], frames["lora_A"] = evaluate(tuned_A, "lora_A")

## 6. 条件 B：拒否例を足して学習し直す（任意）

A の adapter が差し込まれたモデルを捨て、基盤モデルを読み直してから学習します（条件をそろえるため）。

In [ ]:
if CONFIG["run_B"]:
    del tuned_A, base; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    base = load_base()
    tuned_B, train_info["lora_B"] = train_lora(base, train_B, "B")
    summaries["lora_B"], frames["lora_B"] = evaluate(tuned_B, "lora_B")
else:
    print("条件 B は飛ばしました")

## 7. 比較表・退行・提出用 result.json

- `*_rate`：その種類の正解率。`oog_rate` は「用語集に無い」と言えた割合
- `*_over_refusal`：用語集にあるのに「記載がない」と逃げた割合
- `general_rate` / `course_rate`：一般用語 / 講座固有の用語（seen と paraphrase の合計）

In [ ]:
cols = ["seen_rate", "paraphrase_rate", "oog_rate", "control_rate", "general_rate", "course_rate",
        "seen_keyword_recall", "paraphrase_keyword_recall", "seen_over_refusal", "paraphrase_over_refusal", "seconds"]
table = pd.DataFrame(summaries.values()).set_index("label").reindex(columns=cols)
display(table)

# 質問ごとに条件を横に並べる（○/×）。原文は eval_<label>.json
wide = pd.concat({k: f.set_index("question")["ok"].map({True: "○", False: "×"}) for k, f in frames.items()}, axis=1)
wide.insert(0, "type", frames["before"].set_index("question")["type"])
display(wide)

regressed = {k: [q for q in wide.index if wide.loc[q, "before"] == "○" and wide.loc[q, k] == "×"]
             for k in summaries if k.startswith("lora")}
print("退行（学習前は○、学習後に×）:", json.dumps(regressed, ensure_ascii=False, indent=1))

# 用語集に無い語への答え（学習後に“もっともらしく”答えていないか、目で確かめる）
oog_view = pd.concat({k: f[f["type"] == "oog"].set_index("question")["raw"] for k, f in frames.items()}, axis=1)
display(oog_view)

import transformers, peft, trl
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir",)},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）。学習データ: scripts/make_sft_dataset.py（A）、--refusals（B）",
    "summaries": summaries, "train": train_info, "regressed": regressed,
    "executed_training_myself": True,   # 自分で学習を実行できなかった場合は False
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存:", WORK / "result.json", "（eval_*.json も同じ場所）")

## 8. answer.md に書くこと

`/content/glossary_lab` の `result.json` と `eval_*.json` をダウンロードし、比較表を `lectures/02/answer.md` の「宿題」に貼ります。見るところ：

1. **seen と paraphrase の差**：同じ聞き方なら答えられるのに、言い回しを変えると落ちるか（暗記か、知識として使えているか）
2. **general と course の差**：講座固有の語（既定モデル名、lecNN-start タグ、Pi など）は学習前は答えられないはず。学習でどこまで入ったか
3. **oog**：A は「記載がありません」の例を見ていないので、用語集に無い語にも定義っぽく答えるはず。B で直るか、代わりに over_refusal が増えないか
4. **control**：一般知識の質問が定義文の口調になったり、間違えるようになったりしていないか
5. **rag との比較**：検索でヒットした語は rag のほうが正確か。検索が外れた質問（paraphrase の多く、`ハイパーLoRA` など）で何が起きたか

終わったら **ランタイム → ランタイムを接続解除して削除**。